# 03 — Clean human responses
One row per person and question, joined to the question bank. Training and persona evidence only; held-out files are not opened.


In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
PROCESSED = ROOT / "data/processed"

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source = json.loads((ROOT / "configs/data_source.json").read_text())
split = json.loads((ROOT / "data/splits/question_split.json").read_text())
people = json.loads((ROOT / "data/splits/respondent_split.json").read_text())["groups"]
partitions = json.loads((ROOT / "data/splits/partition_manifest.json").read_text())
bank = json.loads((PROCESSED / "question_bank.json").read_text())
bank_audit = json.loads((ROOT / "outputs/question_bank_audit.json").read_text())
assert file_hash(PROCESSED / "question_bank.json") == bank_audit["bank_sha256"]
assert bank["source_revision"] == source["revision"] == split["source_revision"]
assert bank["question_split_sha256"] == file_hash(ROOT / "data/splits/question_split.json")

partition_by_name = {p["partition"]: p for p in partitions["partitions"]}
anchors = [column for group in split["anchor_groups"].values() for column in group]
forbidden = set(split["forbidden_input_columns_all_respondents_all_waves"])
specs = [
    ("shared_training", "shared_training", split["training_columns"], "shared_training_long.parquet"),
    ("development_anchors", "development", anchors, "development_anchors_long.parquet"),
    ("final_anchors", "final_personas", anchors, "final_anchors_long.parquet"),
]


## 1. Question and option lookups
Keep the original option label for checking, and the resolved answer text for modelling.


In [2]:
question_lookup = pd.DataFrame([
    {"item_id": item["item_id"], "parent_key": item["parent_key"], "block": item["block"],
     "question_role": item["role"], "is_anchor": item["is_anchor"],
     "response_format": item["response_format"], "n_options": len(item["options"]),
     "question_text": item["question_text"]}
    for item in bank["items"]
])
option_lookup = pd.DataFrame([
    {"item_id": item["item_id"], "response_code": option["code"],
     "answer_index": option["index"], "source_option_label": option["source_label"],
     "answer_text": option["label"]}
    for item in bank["items"] for option in item["options"]
])
assert question_lookup["item_id"].is_unique
assert not option_lookup.duplicated(["item_id", "response_code"]).any()
assert (option_lookup["answer_index"] == option_lookup["response_code"] - 1).all()
option_lookup["response_code"] = option_lookup["response_code"].astype("Int16")
option_lookup["answer_index"] = option_lookup["answer_index"].astype("Int16")


## 2. Reshape and join
Missing answers stay null. No imputation or reverse-scoring. Repeated wording is stored categorically to keep files small.


In [3]:
def clean_responses(wide, respondent_group):
    records = wide.melt(id_vars="pid", var_name="item_id", value_name="response_code")
    assert not records.duplicated(["pid", "item_id"]).any()
    numeric = pd.to_numeric(records["response_code"], errors="raise")
    assert ((numeric.dropna() % 1) == 0).all()
    records["response_code"] = numeric.astype("Int16")
    records["pid"] = records["pid"].astype("string")
    records["is_missing"] = records["response_code"].isna()

    records = records.merge(question_lookup, on="item_id", how="left", validate="many_to_one", indicator=True)
    assert records["_merge"].eq("both").all(), "Question-bank join failed"
    records = records.drop(columns="_merge")
    records = records.merge(option_lookup, on=["item_id", "response_code"], how="left",
                            validate="many_to_one", indicator=True)
    assert records.loc[~records["is_missing"], "_merge"].eq("both").all(), "Invalid answer code"
    assert records.loc[records["is_missing"], "_merge"].eq("left_only").all()
    records = records.drop(columns="_merge")
    records["respondent_group"] = respondent_group
    records["source_wave"] = "wave1_3"
    records["n_options"] = records["n_options"].astype("int16")

    assert records["answer_index"].isna().equals(records["is_missing"])
    assert records["answer_text"].isna().equals(records["is_missing"])
    assert records["source_option_label"].isna().equals(records["is_missing"])
    assert records["question_role"].eq("training").all()
    assert not set(records["item_id"]) & forbidden
    assert records.loc[~records["is_missing"], "answer_index"].between(
        0, records.loc[~records["is_missing"], "n_options"] - 1).all()

    columns = ["pid", "respondent_group", "source_wave", "item_id", "parent_key", "block",
               "question_role", "is_anchor", "response_format", "question_text", "n_options",
               "response_code", "answer_index", "source_option_label", "answer_text", "is_missing"]
    records = records[columns]
    for column in ["respondent_group", "source_wave", "item_id", "parent_key", "block", "question_role",
                   "response_format", "question_text", "source_option_label", "answer_text"]:
        records[column] = records[column].astype("category")
    return records


## 3. Check against original labelled responses
Read only permitted earlier-wave columns. Check every observed mapping and verify that reshaping preserves the original response table.


In [4]:
label_file = next(f for f in source["files"] if f["repo_path"].endswith("/wave1_3_response_label.csv"))
label_path = ROOT / label_file["local_path"]
assert file_hash(label_path) == label_file["sha256"]

def verify_records(records, wide, ids, columns):
    labelled = pd.read_csv(label_path, usecols=["pid"] + columns, dtype="string",
                           keep_default_na=False, na_values=[""])
    assert labelled["pid"].is_unique
    labelled = labelled.set_index("pid").loc[ids, columns]
    original = labelled.reset_index().melt(id_vars="pid", var_name="item_id", value_name="original_label")
    comparison = records[["pid", "item_id", "source_option_label"]].merge(
        original, on=["pid", "item_id"], how="left", validate="one_to_one", indicator=True)
    assert comparison["_merge"].eq("both").all()
    pd.testing.assert_series_equal(comparison["source_option_label"].astype("string"),
                                   comparison["original_label"], check_names=False)

    restored = records.pivot(index="pid", columns="item_id", values="response_code")
    restored = restored.reindex(index=ids, columns=columns).rename_axis(index="pid", columns=None)
    pd.testing.assert_frame_equal(restored, wide.set_index("pid"), check_dtype=False,
                                   check_index_type=False, check_column_type=False)


## 4. Process and save the three input partitions
Keep missing rows for audit. Training will use only rows where `is_missing` is false.


In [5]:
summaries = []
preview = None
for partition_name, respondent_group, columns, output_name in specs:
    spec = partition_by_name[partition_name]
    input_path = ROOT / spec["path"]
    assert input_path.parent == PROCESSED
    assert spec["purpose"] in {"training", "persona_evidence"}
    assert file_hash(input_path) == spec["sha256"]
    assert not set(columns) & forbidden
    ids = people[respondent_group]

    wide = pd.read_parquet(input_path)
    assert wide["pid"].tolist() == ids and wide["pid"].is_unique
    assert wide.columns.tolist() == ["pid"] + columns
    records = clean_responses(wide, respondent_group)
    verify_records(records, wide, ids, columns)
    assert len(records) == len(ids) * len(columns)
    assert int(records["is_missing"].sum()) == spec["missing_answers"]
    if partition_name != "shared_training":
        assert records["is_anchor"].all()

    output_path = PROCESSED / output_name
    records.to_parquet(output_path, index=False, compression="zstd", row_group_size=100_000)
    saved = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(saved, records)
    missing_by_item = records.loc[records["is_missing"]].groupby("item_id", observed=True).size()
    summaries.append({
        "partition": partition_name, "people": len(ids), "questions": len(columns),
        "rows": len(records), "observed_answers": int((~records["is_missing"]).sum()),
        "missing_answers": int(records["is_missing"].sum()),
        "missing_by_item": {str(k): int(v) for k, v in missing_by_item.items()},
        "input_path": str(input_path.relative_to(ROOT)), "input_sha256": spec["sha256"],
        "output_path": str(output_path.relative_to(ROOT)), "output_sha256": file_hash(output_path),
    })
    if partition_name == "shared_training":
        preview = records.loc[(records["pid"] == ids[0]) & records["item_id"].isin(
            ["QID84_4", "QID250_1", "QID119"]),
            ["pid", "item_id", "response_code", "answer_index", "source_option_label", "answer_text"]].copy()
    del wide, records, saved

display(pd.DataFrame(summaries)[["partition", "people", "questions", "rows", "observed_answers", "missing_answers"]])
display(preview)


,partition,people,questions,rows,observed_answers,missing_answers
0,shared_training,1658,430,712940,712827,113
1,development_anchors,300,92,27600,27576,24
2,final_anchors,100,92,9200,9198,2


,pid,item_id,response_code,answer_index,source_option_label,answer_text
308388,95,QID84_4,2,1,2,$3.00 in 5 weeks
407868,95,QID250_1,2,1,2,$0.50
573668,95,QID119,7,6,I would keep $6 for myself and send $6 to the ...,I would keep $6 for myself and send $6 to the ...


## 5. Save the audit
The existing data, question bank and split assignments are unchanged. Held-out labels remain in their separate files.


In [6]:
assert not set(people["shared_training"]) & (set(people["development"]) | set(people["final_personas"]))
assert not set(people["development"]) & set(people["final_personas"])
assert file_hash(PROCESSED / "question_bank.json") == bank_audit["bank_sha256"]
for summary in summaries:
    assert file_hash(ROOT / summary["input_path"]) == summary["input_sha256"]
assert file_hash(label_path) == label_file["sha256"]

audit = {
    "status": "passed", "source_revision": source["revision"],
    "question_bank_sha256": bank_audit["bank_sha256"],
    "labelled_source_sha256": label_file["sha256"],
    "partitions": summaries,
    "response_code_base": 1, "answer_index_base": 0,
    "missing_value_policy": "Preserved as null rows with is_missing=True; exclude these rows when fitting. Missingness reasons are not inferred.",
    "checks": ["fixed participant membership", "allowed question columns", "unique person/item rows",
               "many-to-one question and option joins", "original labelled-source agreement",
               "lossless wide-to-long reshaping", "missing values preserved", "Parquet round-trip equality"],
    "heldout_response_files_loaded": False,
    "training_started": False,
}
(ROOT / "outputs/response_cleaning_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
print("Passed all checks. Saved three joined response tables and the audit report.")


Passed all checks. Saved three joined response tables and the audit report.
